# Qwen3-4B 金融知识微调（Google Colab）

本 Notebook 用于在 Google Colab 上微调 Qwen3-4B 模型，提升金融领域知识能力，同时保持通用知识能力。

## 结构
- **公共部分**：环境安装 → Drive 挂载 → 数据准备 → 模型加载
- **方案一**：transformers/peft 微调 → 自写评估脚本对比
- **方案二**：LLaMA-Factory 微调 → EvalScope 评估

---
# 公共部分：环境、数据、模型（两个方案共用）

## 0. 环境安装

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets torch scikit-learn tqdm

In [4]:
import torch
import os
import json
import time
from datetime import datetime


def log_gpu_memory(tag=""):
    if torch.cuda.is_available():
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        usage_pct = reserved / total * 100
        print(f"[GPU] {tag} | 已分配: {allocated:.1f}GB | 已预留: {reserved:.1f}GB / {total:.1f}GB ({usage_pct:.0f}%)")
    else:
        print(f"[GPU] {tag} | CUDA 不可用")


print("="*60)
print(f"环境检查 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)
print(f"PyTorch:     {torch.__version__}")
print(f"CUDA 可用:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 名称:    {torch.cuda.get_device_name(0)}")
    print(f"GPU 总显存:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
log_gpu_memory("初始状态")

环境检查 | 2026-05-20 08:31:39
PyTorch:     2.10.0+cu128
CUDA 可用:   True
GPU 名称:    Tesla T4
GPU 总显存:  14.6 GB
[GPU] 初始状态 | 已分配: 0.0GB | 已预留: 0.0GB / 14.6GB (0%)


## 1. 挂载 Google Drive + 准备数据

In [ ]:
from google.colab import drive
# 固定的挂载点
MOUNT_POINT = '/content/mydrive'

# 检查是否已经挂载
if not os.path.exists(MOUNT_POINT):
    os.makedirs(MOUNT_POINT, exist_ok=True)
    
# 尝试挂载（如果已挂载会快速跳过）
try:
    drive.mount(MOUNT_POINT, force_remount=False)
except:
    print("请手动授权挂载")

WORK_DIR = f'{MOUNT_POINT}/qwen3_finetune'
os.makedirs(WORK_DIR, exist_ok=True)

print(f"[Drive] 工作目录: {WORK_DIR}")


Mounted at /content/drive1


In [ ]:
from google.colab import drive




[Drive] 工作目录: /content/drive/MyDrive/qwen3_finetune


In [10]:
# 从 GitHub 克隆项目（含训练数据和测试数据），然后复制到 Drive 持久化
REPO_DIR = '/content/qwen3-4b-finance-finetune'
DATA_ON_DRIVE = f'{WORK_DIR}/finetune_data_final'

# 步骤1: 克隆 GitHub 项目到临时目录
if not os.path.exists(REPO_DIR):
    print("[数据] 从 GitHub 克隆项目...")
    !git clone https://github.com/jslijb/qwen3-4b-finance-finetune.git {REPO_DIR}
else:
    print(f"[数据] 项目已存在: {REPO_DIR}")

# 步骤2: 复制数据到 Drive（持久化，跨会话不丢失）
if not os.path.exists(DATA_ON_DRIVE):
    print("\n[数据] 首次运行：复制数据到 Drive...")
    import shutil
    shutil.copytree(f'{REPO_DIR}/finetune_data_final', DATA_ON_DRIVE)
    print(f"[数据] 已复制到 Drive: {DATA_ON_DRIVE}")
else:
    print(f"\n[数据] Drive 中已存在数据: {DATA_ON_DRIVE}")

# 步骤3: 验证数据文件
data_files = {
    'train.json': '训练集',
    'mmlu_test.json': 'MMLU 测试集',
    'ceval_test.json': 'C-Eval 测试集',
    'finance_test.json': 'Fin-Eva 测试集',
    'dataset_info.json': 'LLaMA-Factory 配置',
}

print("\n[数据] 文件检查:")
for fname, desc in data_files.items():
    fpath = os.path.join(DATA_ON_DRIVE, fname)
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f:
            count = sum(1 for _ in f)
        print(f"  ✓ {fname:<25} ({desc}): {count:>5} 条")
    else:
        print(f"  ✗ {fname:<25} ({desc}): 不存在")

[数据] 从 GitHub 克隆项目...
Cloning into '/content/qwen3-4b-finance-finetune'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 21 (delta 7), reused 17 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 2.01 MiB | 6.04 MiB/s, done.
Resolving deltas: 100% (7/7), done.

[数据] 首次运行：复制数据到 Drive...
[数据] 已复制到 Drive: /content/drive/MyDrive/qwen3_finetune/finetune_data_final

[数据] 文件检查:
  ✓ train.json                (训练集):  2000 条
  ✓ mmlu_test.json            (MMLU 测试集):   500 条
  ✓ ceval_test.json           (C-Eval 测试集):   500 条
  ✓ finance_test.json         (Fin-Eva 测试集):  1000 条
  ✗ dataset_info.json         (LLaMA-Factory 配置): 不存在


## 2. 加载模型和 Tokenizer（公共，两个方案共用）

模型缓存到 Google Drive（`HF_HOME`），首次下载后跨会话复用，无需重新下载。

In [11]:
# 设置 HuggingFace 缓存目录到 Drive（模型持久化，首次下载后跨会话复用）
MODEL_CACHE_DIR = f'{WORK_DIR}/model_cache'
os.environ['HF_HOME'] = MODEL_CACHE_DIR
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

# 检查模型是否已缓存到 Drive
_model_hub_dir = os.path.join(MODEL_CACHE_DIR, 'hub', 'models--Qwen--Qwen3-4B')
if os.path.exists(_model_hub_dir):
    print(f"[模型] ✓ 已在 Drive 中找到缓存: {MODEL_CACHE_DIR}")
else:
    print(f"[模型] 首次运行：将从 HuggingFace 下载并缓存到 Drive...")
    print(f"[模型] 缓存目标: {MODEL_CACHE_DIR}")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# 公共配置（所有方案共用）
MODEL_NAME = "Qwen/Qwen3-4B"
MAX_LENGTH = 2048

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("[模型] 加载 Tokenizer...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"[模型] Tokenizer 就绪 (padding_side=left)")

print("[模型] 加载原模型（4-bit 量化）...")
original_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print(f"[模型] 参数量: {original_model.num_parameters() / 1e9:.2f}B")
print(f"[模型] 加载耗时: {time.time()-t0:.1f}s")
log_gpu_memory("模型加载完成")

# 统计缓存大小
import subprocess as _sp
try:
    _result = _sp.run(['du', '-sh', MODEL_CACHE_DIR], capture_output=True, text=True)
    print(f"[模型] 缓存大小: {_result.stdout.strip()}")
except Exception:
    pass

[模型] 加载 Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[模型] 参数量: 4.02B
[模型] 加载耗时: 171.9s
[GPU] 模型加载完成 | 已分配: 2.5GB | 已预留: 6.9GB / 14.6GB (47%)


---

# 方案一：transformers/peft 微调 + 自写评估对比

使用原生 HuggingFace API 进行微调和评估。

## A1. 评估原模型（微调前基线）

In [ ]:
def evaluate_model(model, tokenizer, test_data_path, dataset_name="", eval_batch_size=16):
    from tqdm import tqdm
    
    with open(test_data_path, 'r', encoding='utf-8') as f:
        test_data = [json.loads(line) for line in f]
    
    correct = 0
    total = len(test_data)
    model.eval()
    
    prompts = []
    true_answers = []
    for item in test_data:
        messages = item["messages"]
        user_msg = ""
        true_answer = ""
        for msg in messages:
            if msg["role"] == "user":
                user_msg = msg["content"]
            elif msg["role"] == "assistant":
                true_answer = msg["content"]
        prompts.append(f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n")
        true_answers.append(true_answer)
    
    t_start = time.time()
    predicted_answers = []
    n_batches = (total + eval_batch_size - 1) // eval_batch_size
    for i in tqdm(range(n_batches), desc=f"评估{dataset_name}", leave=False):
        batch_prompts = prompts[i * eval_batch_size : (i + 1) * eval_batch_size]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH - 10).to(model.device)
        input_lens = inputs["attention_mask"].sum(dim=1)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.pad_token_id,
            )
        
        for j in range(len(batch_prompts)):
            actual_len = input_lens[j].item()
            generated = tokenizer.decode(outputs[j][actual_len:], skip_special_tokens=True)
            predicted = generated.strip()[0] if generated.strip() else ""
            predicted_answers.append(predicted)
    
    for pred, true in zip(predicted_answers, true_answers):
        if pred.upper() == true.upper():
            correct += 1
    
    elapsed = time.time() - t_start
    accuracy = correct / total if total > 0 else 0
    speed = total / elapsed if elapsed > 0 else 0
    print(f"  [结果] {dataset_name}: 准确率={accuracy:.4f} ({correct}/{total}) | 耗时={elapsed:.1f}s | 速度={speed:.1f}条/s | batch={eval_batch_size}")
    return accuracy, correct, total


def run_evaluation(model, tokenizer, data_dir, tag="", eval_batch_size=16):
    test_datasets = {
        'MMLU': f'{data_dir}/mmlu_test.json',
        'C-Eval': f'{data_dir}/ceval_test.json',
        'Fin-Eva': f'{data_dir}/finance_test.json',
    }
    results = {}
    for name, path in test_datasets.items():
        if os.path.exists(path):
            accuracy, correct, total = evaluate_model(model, tokenizer, path, name, eval_batch_size)
            results[name] = accuracy
        else:
            print(f"  [跳过] {name}: 数据不存在")
    log_gpu_memory(f"{tag}评估完成")
    return results


print("="*60)
print("[方案一] 评估原模型（微调前基线）")
print("="*60)
baseline_results = run_evaluation(original_model, tokenizer, DATA_ON_DRIVE, tag="原模型", eval_batch_size=16)

## A2. 使用 peft 微调

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset

OUTPUT_DIR = '/content/output_peft'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
print("[微调] 释放原模型显存...")
del original_model
torch.cuda.empty_cache()
log_gpu_memory("原模型已释放")

print("\n[微调] 重新加载模型用于微调...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=4, lora_alpha=8, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"[微调] 耗时: {time.time()-t0:.1f}s")
log_gpu_memory("微调模型就绪")

In [ ]:
TRAIN_DATA_PATH = f'{DATA_ON_DRIVE}/train.json'
with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as f:
    train_data = [json.loads(line) for line in f]
dataset = Dataset.from_list(train_data)
print(f"[数据] 训练集大小: {len(dataset)} 条")

In [ ]:
def preprocess_function(examples):
    inputs = []
    targets = []
    for messages in examples["messages"]:
        user_msg = ""
        assistant_msg = ""
        for msg in messages:
            if msg["role"] == "user": user_msg = msg["content"]
            elif msg["role"] == "assistant": assistant_msg = msg["content"]
        inputs.append(f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n")
        targets.append(assistant_msg)
    
    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH - 50, truncation=True, padding=False, return_tensors=None)
    labels = tokenizer(targets, max_length=50, truncation=True, padding=False, return_tensors=None)
    
    result_input_ids, result_attention_mask, result_labels = [], [], []
    for i in range(len(inputs)):
        inp_ids = model_inputs["input_ids"][i]
        lbl_ids = labels["input_ids"][i]
        combined_ids = inp_ids + lbl_ids + [tokenizer.eos_token_id]
        combined_mask = [1] * len(combined_ids)
        combined_labels = [-100] * len(inp_ids) + lbl_ids + [tokenizer.eos_token_id]
        result_input_ids.append(combined_ids)
        result_attention_mask.append(combined_mask)
        result_labels.append(combined_labels)
    return {"input_ids": result_input_ids, "attention_mask": result_attention_mask, "labels": result_labels}

print("[数据] 预处理中...")
t0 = time.time()
processed_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names, desc="预处理")
split_dataset = processed_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"[数据] 预处理耗时: {time.time()-t0:.1f}s | 训练集: {len(train_dataset)} | 验证集: {len(eval_dataset)}")

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=1e-5, weight_decay=0.01, num_train_epochs=1,
    lr_scheduler_type="cosine", warmup_ratio=0.1, bf16=True, fp16=False,
    logging_steps=10, save_steps=100, save_total_limit=2, eval_strategy="steps", eval_steps=100,
    gradient_checkpointing=True, optim="adamw_torch", report_to="none", seed=42,
)

effective_bs = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print("="*60)
print("[方案一] 训练参数")
print("="*60)
print(f"  学习率: {training_args.learning_rate} | 批次: {training_args.per_device_train_batch_size}x{training_args.gradient_accumulation_steps}={effective_bs}")
print(f"  轮数: {training_args.num_train_epochs} | LoRA Rank: {lora_config.r} | Alpha: {lora_config.lora_alpha}")
log_gpu_memory("训练配置就绪")

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, max_length=MAX_LENGTH)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=eval_dataset, data_collator=data_collator,
)

print("="*60)
print("[方案一] 开始微调...")
print(f"[时间] 开始: {datetime.now().strftime('%H:%M:%S')}")
print("="*60)
t_train_start = time.time()
trainer.train()
t_train_end = time.time()
print(f"\n[时间] 结束: {datetime.now().strftime('%H:%M:%S')}")
print(f"[耗时] {(t_train_end-t_train_start)/60:.1f}min")
log_gpu_memory("训练完成")

In [ ]:
lora_output_dir = f"{OUTPUT_DIR}/lora_final"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
import glob
adapter_files = glob.glob(os.path.join(lora_output_dir, '*'))
total_size = sum(os.path.getsize(f) for f in adapter_files) / 1024**2
print(f"[保存] LoRA 权重: {lora_output_dir} | 文件: {len(adapter_files)} | 大小: {total_size:.1f}MB")

## A3. 评估微调后模型 + 对比

In [ ]:
print("="*60)
print("[方案一] 评估微调后模型")
print("="*60)
finetuned_results = run_evaluation(model, tokenizer, DATA_ON_DRIVE, tag="微调后", eval_batch_size=16)

In [ ]:
print("="*70)
print("[方案一] 微调前后对比")
print("="*70)
print(f"{'数据集':<10} {'微调前':<10} {'微调后':<10} {'变化':<10} {'结论'}")
print("-" * 65)

for name in baseline_results:
    before = baseline_results[name]
    after = finetuned_results.get(name, 0)
    diff = after - before
    sign = "+" if diff > 0 else ""
    if diff >= -0.02 and name != 'Fin-Eva': verdict = "✓ 保持"
    elif diff > 0 and name == 'Fin-Eva': verdict = "✓ 提升"
    else: verdict = "✗ 下降"
    print(f"{name:<10} {before:<10.4f} {after:<10.4f} {sign}{diff:<9.4f} {verdict}")

if baseline_results and finetuned_results:
    before_avg = sum(baseline_results.values()) / len(baseline_results)
    after_avg = sum(finetuned_results.values()) / len(finetuned_results)
    diff_avg = after_avg - before_avg
    sign = "+" if diff_avg > 0 else ""
    print("-" * 65)
    print(f"{'平均':<10} {before_avg:<10.4f} {after_avg:<10.4f} {sign}{diff_avg:<9.4f}")
    
    fin_eva_diff = finetuned_results.get('Fin-Eva', 0) - baseline_results.get('Fin-Eva', 0)
    mmlu_diff = finetuned_results.get('MMLU', 0) - baseline_results.get('MMLU', 0)
    ceval_diff = finetuned_results.get('C-Eval', 0) - baseline_results.get('C-Eval', 0)
    
    print(f"\n[结论]")
    print(f"  金融(Fin-Eva):  {'+' if fin_eva_diff > 0 else ''}{fin_eva_diff:+.4f}  {'✓ 提升成功' if fin_eva_diff > 0 else '✗ 未提升'}")
    print(f"  通用(MMLU):    {'+' if mmlu_diff > 0 else ''}{mmlu_diff:+.4f}  {'✓ 保持良好' if mmlu_diff >= -0.02 else '✗ 下降较多'}")
    print(f"  中文(C-Eval):  {'+' if ceval_diff > 0 else ''}{ceval_diff:+.4f}  {'✓ 保持良好' if ceval_diff >= -0.02 else '✗ 下降较多'}")

---

# 方案二：LLaMA-Factory 微调 + EvalScope 评估

与本地服务器完全一致的工具链。数据来自公共部分的 `DATA_ON_DRIVE`。

## B1. 安装 LLaMA-Factory 和 EvalScope

In [ ]:
!pip install -q llama-factory evalscope vllm
print("[安装] 完成")

## B2. 使用 LLaMA-Factory 微调

In [ ]:
import yaml

llamafactory_config = {
    "model_name_or_path": MODEL_NAME,
    "stage": "sft", "do_train": True, "finetuning_type": "lora",
    "lora_rank": 4, "lora_alpha": 8, "lora_target": "all",
    "dataset": "finetune_train", "dataset_dir": DATA_ON_DRIVE, "template": "qwen3", "cutoff_len": 2048,
    "output_dir": f'{WORK_DIR}/saves/qwen3-4b-finance-lora',
    "per_device_train_batch_size": 4, "per_device_eval_batch_size": 4, "gradient_accumulation_steps": 4,
    "learning_rate": 1e-5, "num_train_epochs": 1, "lr_scheduler_type": "cosine", "warmup_ratio": 0.1,
    "bf16": True, "logging_steps": 10, "save_steps": 100, "save_total_limit": 2,
    "gradient_checkpointing": True, "optim": "adamw_torch", "report_to": "none", "seed": 42,
}

config_path = f'{WORK_DIR}/llama_factory_sft.yaml'
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(llamafactory_config, f, allow_unicode=True, default_flow_style=False)

print(f"[配置] 已生成: {config_path}")
for k, v in llamafactory_config.items():
    print(f"  {k}: {v}")

In [ ]:
print("="*60)
print("[方案二] LLaMA-Factory 开始微调")
print(f"[时间] 开始: {datetime.now().strftime('%H:%M:%S')}")
print("="*60)
log_gpu_memory("微调前")
t0 = time.time()
!cd / && llamafactory-cli train {config_path}
print(f"\n[时间] 结束: {datetime.now().strftime('%H:%M:%S')} | 耗时: {(time.time()-t0)/60:.1f}min")
log_gpu_memory("微调完成")

## B3. 合并 LoRA 权重

In [ ]:
MERGED_OUTPUT = f'{WORK_DIR}/saves/qwen3-4b-finance-merged'
LORA_PATH = f'{WORK_DIR}/saves/qwen3-4b-finance-lora'

export_config = {
    "model_name_or_path": MODEL_NAME, "adapter_name_or_path": LORA_PATH,
    "template": "qwen3", "finetuning_type": "lora",
    "export_dir": MERGED_OUTPUT, "export_size": 2, "export_device": "cpu",
}

export_path = f'{WORK_DIR}/llama_factory_export.yaml'
with open(export_path, 'w', encoding='utf-8') as f:
    yaml.dump(export_config, f, allow_unicode=True, default_flow_style=False)

print("[合并] 导出配置:")
for k, v in export_config.items():
    print(f"  {k}: {v}")

!cd / && llamafactory-cli export {export_path}
print(f"[合并] 完成: {MERGED_OUTPUT}")

## B4. 启动 vLLM 服务

In [ ]:
VLLM_PORT = 8000

print("[部署] 启动 vLLM 服务...")
import subprocess
vllm_proc = subprocess.Popen(
    ['CUDA_LAUNCH_BLOCKING=1', 'vllm', 'serve', MERGED_OUTPUT,
     '--served-model-name', 'qwen3-4b-finance',
     '--max-model-len', '4096', '--max-num-seqs', '85',
     '--port', str(VLLM_PORT), '--enforce-eager', '--enable-prefix-caching'],
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

import time, urllib.request
print("[部署] 等待服务启动...")
time.sleep(30)
try:
    req = urllib.request.Request(f'http://127.0.0.1:{VLLM_PORT}/v1/models')
    resp = urllib.request.urlopen(req, timeout=5)
    print(f"[部署] ✓ 服务已启动: {resp.read().decode()}")
except Exception as e:
    print(f"[部署] ✗ 未就绪: {e}（可能需要等待更长时间）")

## B5. 使用 EvalScope 评估

In [ ]:
API_URL = f'http://127.0.0.1:{VLLM_PORT}/v1'

def run_evalscope(subset_list, eval_name=""):
    subset_str = ", ".join(subset_list)
    args_json = json.dumps({"local_path": DATA_ON_DRIVE, "subset_list": subset_list})
    gen_json = json.dumps({"do_sample": False, "temperature": 0})
    cmd = (
        f'evalscope eval --model qwen3-4b-finance --api-url {API_URL} --api-key EMPTY '
        f'--eval-batch-size 5 --datasets general_mcq '
        f'--dataset-args "{args_json}" --generation-config "{gen_json}" --seed 42'
    )
    print(f"\n[EvalScope] 评估: {eval_name} ({subset_str})")
    t0 = time.time()
    !{cmd}
    print(f"[EvalScope] 耗时: {time.time()-t0:.1f}s")


print("="*60)
print("[方案二] EvalScope 开始评估")
print("="*60)
run_evalscope(['mmlu_test'], 'MMLU')
run_evalscope(['ceval_test'], 'C-Eval')
run_evalscope(['finance_test'], 'Fin-Eva')
run_evalscope(['mmlu_test', 'ceval_test', 'finance_test'], '综合')

## B6. 停止服务 + 汇总

In [ ]:
if 'vllm_proc' in dir():
    print("[停止] 关闭 vLLM...")
    vllm_proc.terminate()
    vllm_proc.wait(timeout=10)
    print("[停止] 已关闭")

log_gpu_memory("最终释放")
print(f"\n[Drive] 工作目录: {WORK_DIR}")
print(f"[Drive] 数据目录: {DATA_ON_DRIVE}")
print(f"[Drive] 方案一输出: {OUTPUT_DIR}")
print(f"[Drive] 方案二LoRA: {LORA_PATH}")
print(f"[Drive] 方案二合并: {MERGED_OUTPUT}")
print(f"\n[时间戳] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")